In [24]:

from pathlib import Path

import pandas as pd
from nhs_waiting_lists import (
    __app_name__,
)
from nhs_waiting_lists.constants import DB_FILE
from nhs_waiting_lists.constants import proj_db_path
from nhs_waiting_lists.utils.xdg import XDGBasedir
from sqlalchemy import create_engine
from sqlalchemy import text, bindparam

project_root = Path(XDGBasedir.get_data_dir(__app_name__))

In [25]:


DB_PATH = project_root / proj_db_path / DB_FILE
DATA_DIR = "./data"

engine = create_engine(f"sqlite:///{DB_PATH}")

In [26]:
PROVIDER_CODES = [
    "R0B",
    "RAJ",
    "RDE",
    "RDU",
    "REF",
    "RGN",
    "RH8",
    "RHU",
    "RHW",
    "RJ2",
    "RL4",
    "RN5",
    "RTE",
    "RTF",
    "RVJ",
    "RWD",
    "RWF",
    "RWH",
    "RWP",
    "RWY",
    "RXC",
    "RXK",
    "RXR",
]
TREATMENT_CODES = (
    'C_101',
    'C_110',
    'C_301',
    'C_320',
    'C_330',
    'C_400',
    'C_502',
)

In [27]:


query = text("""
             SELECT period,
                    provider       AS provider,
                    pathway,
                    treatment AS treatment,
                    total_all
             FROM all_rtt_raw
             WHERE provider IN :provider_codes
               AND treatment IN :treatment_codes
             ORDER BY provider ASC, treatment ASC, period ASC; \
             """).bindparams(
    bindparam('provider_codes', expanding=True),
    bindparam('treatment_codes', expanding=True)
)

conn = engine.raw_connection()

df = pd.read_sql(query, engine, params={
    'provider_codes': PROVIDER_CODES,
    'treatment_codes': TREATMENT_CODES}
                 )  # type: ignore[arg-type]
df


,period,provider,pathway,treatment,total_all
0,2022-12,R0B,admitted,C_101,1
1,2022-12,R0B,nonadmitted,C_101,2
2,2022-12,R0B,incomplete,C_101,3
3,2022-12,R0B,incomplete_dta,C_101,2
4,2022-12,R0B,incomplete,C_101,1
...,...,...,...,...,...
268061,2025-09,RXR,incomplete_dta,C_502,1
268062,2025-09,RXR,new_periods,C_502,1
268063,2025-09,RXR,incomplete,C_502,3
268064,2025-09,RXR,incomplete_dta,C_502,1


In [29]:
df.query('provider == "R0B" and treatment == "C_320"')[
    [
        "period",
        "incomplete",
        "incomplete_prev",
        "incomplete_diff",
        "residual",
        "delta",
        "residual_check"
    ]
]

KeyError: "['incomplete', 'incomplete_prev', 'incomplete_diff', 'residual', 'delta', 'residual_check'] not in index"

In [ ]:
from plotnine import (
    scale_x_datetime
)

df2 = df.query('provider == "R0B" and treatment_code == "C_301"')[
    [
        "period",
        "incomplete",
        "incomplete_prev",
        "incomplete_diff",
        "residual",
        "admitted",
        "nonadmitted",
        "new_periods",
        "delta",
    ]
]

df2

In [ ]:
import statsmodels.api as sm

df3 = df2.dropna()

X = sm.add_constant(df3['delta'])
y = df3['residual']
ols = sm.OLS(y, X).fit(cov_type='HC3')
print(ols.params, ols.rsquared)
print(ols.summary2())

In [ ]:

child_long = df2.reset_index().melt(
    id_vars=['period'],
    value_vars=['residual', 'delta', 'incomplete_diff', 'new_periods', 'nonadmitted', 'admitted'],
    var_name='metric',
    value_name='value'
)


# child_long = child_long.groupby('metric').apply(
#     lambda x: x[(x['value'] >= x['value'].quantile(0.05)) &
#                 (x['value'] <= x['value'].quantile(0.95))],
#     include_groups=True
# ).reset_index(drop=True)

def remove_outliers(group: pd.DataFrame) -> pd.DataFrame:
    Q1 = group['value'].quantile(0.25)
    Q3 = group['value'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return group[(group['value'] >= lower) & (group['value'] <= upper)]


child_long = child_long.groupby('metric').apply(remove_outliers).reset_index(drop=True)

child_long

In [ ]:
from plotnine import ggplot, aes, geom_col, facet_wrap, theme_bw, theme, element_text

p = (
        ggplot(child_long, aes(x="period", y="value", fill="metric"))
        + scale_x_datetime(date_labels="%y-%m")
        + geom_col(position="dodge")  # side-by-side bars
        + facet_wrap("~metric", ncol=1)
        + theme_bw()
        + theme(axis_text_x=element_text(rotation=45, hjust=1))
)
p



In [ ]:
from mizani.formatters import percent_format
from plotnine import (
    ggplot, aes, geom_histogram, geom_vline, geom_qq,
    labs, scale_x_continuous, facet_wrap, theme_minimal,
    stat_qq_line, geom_density
)

quantile_05 = df2["delta"].quantile(0.05)

apple_returns_figure = (
        ggplot(df2, aes(x="delta"))
        + geom_histogram(bins=100)
        + geom_vline(aes(xintercept=quantile_05), linetype="dashed")
        + labs(x="", y="", title="Distribution of daily Apple stock returns")

)
apple_returns_figure.show()

In [ ]:
child_long.groupby("metric")["value"].quantile(0.05).reset_index()

In [ ]:

quantiles = child_long.groupby("metric")["value"].quantile(0.05).reset_index()
quantiles.columns = ["return_type", "q05"]
quantiles

In [ ]:

from nhs_waiting_lists.utils.normality import test_normality

returns_comparison = (
        ggplot(child_long, aes(x="value"))
        + geom_histogram(aes(y="..density.."), bins=50, alpha=0.7, fill="lightblue")
        + geom_density(color="red", size=1)
        + geom_vline(
    data=quantiles,
    mapping=aes(xintercept="q05"),
    linetype="dashed",
    color="orange"
)
        + facet_wrap("metric", scales="free", labeller=lambda x: {
    "delta": "Activity delta",
    "admitted": "admitted stops",
    "nonadmitted": "non admitted stops",
    "inc_diff": "Incomplete diff",
    "new_periods": "new strats",
    "residual": "residuals",
}[x])
        + labs(
    x="Return Value",
    y="Density",
    title="Distribution Comparison: Percentage vs Log Returns"
)
        + scale_x_continuous(labels=percent_format())
        + theme_minimal()
)

# For Q-Q plots to test normality more directly
qq_plot = (
        ggplot(child_long, aes(sample="value"))
        + geom_qq()
        + stat_qq_line(color="red")
        + facet_wrap("metric", scales="free", labeller=lambda x: {
    "delta": "Activity delta",
    "admitted": "admitted stops",
    "nonadmitted": "non admitted stops",
    "inc_diff": "Incomplete diff",
    "new_periods": "new strats",
    "residual": "residuals",
}[x])
        + labs(
    x="Theoretical Quantiles",
    y="Sample Quantiles",
    title="Q-Q Plots: Testing Normality Assumptions"
)
        + theme_minimal()
)

returns_comparison.show()
qq_plot.show()

pct_results = test_normality(df2["delta"].dropna(), "Delta")
log_results = test_normality(df2["residual"].dropna(), "Residual")
nap_results = test_normality(df2["nonadmitted"].dropna(), "Non Admitted")
ap_results = test_normality(df2["admitted"].dropna(), "Admitted")
np_results = test_normality(df2["new_periods"].dropna(), "New Periods")

normality_df = pd.DataFrame([pct_results, log_results, nap_results, ap_results, np_results])
print("\nNormality Test Results:")
print(normality_df.round(4))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import pandas as pd

provider = "R0B"
treatment = "C_301"

df3 = df2.dropna(subset=['delta', 'residual']).copy()

# Fit OLS
X = sm.add_constant(df3['delta'])
y = df3['residual']
ols = sm.OLS(y, X).fit(cov_type='HC3')

# Predictions
xvals = np.linspace(df3['delta'].min(), df3['delta'].max(), 200)
yhat = ols.params['const'] + ols.params['delta'] * xvals

# Parse period and flag COVID
df3['period'] = pd.to_datetime(df3['period'])
covid = (df3['period'].dt.year == 2020)

# --- Plot starts here ---
plt.figure(figsize=(8, 6))

# Scatter: two colours in same figure
plt.scatter(df3.loc[~covid, 'delta'], df3.loc[~covid, 'residual'],
            alpha=0.6, color='steelblue', label='Normal months')
plt.scatter(df3.loc[covid, 'delta'], df3.loc[covid, 'residual'],
            alpha=0.8, color='firebrick', label='COVID months')

# Regression line
plt.plot(xvals, yhat, color='darkorange', lw=2,
         label=f"OLS fit: y = {ols.params['delta']:.2f}x + {ols.params['const']:.1f}")

# Formatting
plt.axhline(0, color='grey', lw=1)
plt.axvline(0, color='grey', lw=1)
plt.xlabel("Δ (New - Completed pathways)", fontsize=11)
plt.ylabel("Residual (Untracked change in incompletes)", fontsize=11)
plt.title(f"{provider} {treatment}: Residual vs Δ\nR²={ols.rsquared:.2f}, β={ols.params['delta']:.2f}", fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

x = np.concatenate(([5, 6, 7, 8, 9, 10], [1, 2, 3, 4]))
y = np.random.rand(10)

plt.figure(figsize=(10, 6))
plt.scatter(df3["delta"], df3["residual"])
plt.xlabel('delta')
plt.ylabel('residual')
plt.title('Scatter Plot with Discontinuous X-axis')
plt.show()

In [ ]:
from plotnine import geom_line

p = (
        ggplot(child_long, aes(x="period", y="value", group=1))
        + scale_x_datetime(date_labels="%y-%m")
        + geom_line()
        + facet_wrap("~metric", ncol=1, scales="free_y")  # or ncol=2 for side-by-side
        + theme_bw()
)
p

In [ ]:
p = (
        ggplot(child_long, aes(x="period", y="value", fill="metric"))
        + scale_x_datetime(date_labels="%y-%m")
        + geom_col(position="dodge")  # or position="stack" for stacked
        + theme_bw()
        + theme(axis_text_x=element_text(rotation=45, hjust=1))
)
p

In [ ]:
returns_wide = (
    df.query("treatment_code == 'C_101'")
    .pivot(
        index="period",
        columns="provider",
        values="frac_change"
    )
    .reset_index()
)

returns_wide